<a href="https://colab.research.google.com/github/Shibin2000/covid19-etl-pipeline/blob/main/COVID_19_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COVID-19 Data Engineering Pipeline

End-to-end pipeline using Pandas, PySpark, DuckDB, and Plotly.

Covers ingestion, transformation, distributed processing, storage, and visualization.

In [57]:
!pip install duckdb plotly pandas pyspark

## Data Ingestion

In [58]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"

def load_data(url: str) -> pd.DataFrame:
    df = pd.read_csv(url)
    print(f"Loaded {len(df)} rows | {df['location'].nunique()} countries")
    return df

df_raw = load_data(DATA_URL)

Loaded 429435 rows | 255 countries


## Data Transformation

In [59]:

def transform_data(df: pd.DataFrame) -> pd.DataFrame:
    cols = [
        'location', 'date', 'total_cases', 'total_deaths',
        'new_cases', 'new_deaths', 'population',
        'total_vaccinations', 'continent'
    ]

    countries = [
        'United States', 'India', 'Brazil', 'United Kingdom',
        'France', 'Germany', 'Italy', 'Canada', 'Australia'
    ]

    df = df[cols].copy()
    df = df[df['location'].isin(countries)]

    num_cols = df.select_dtypes(include=['number']).columns
    df[num_cols] = df[num_cols].fillna(0)

    df['date'] = pd.to_datetime(df['date'])
    df['death_rate'] = (
        df['total_deaths'] / df['total_cases'].replace(0, pd.NA) * 100
    ).clip(upper=100).fillna(0).round(2)

    df = df[df['total_cases'] > 1000]

    return df

df = transform_data(df_raw)
print(df.shape)
print(df.head())

(14331, 10)
        location       date  total_cases  total_deaths  new_cases  new_deaths  \
21853  Australia 2020-03-22       1081.0           9.0      832.0         2.0   
21854  Australia 2020-03-23       1081.0           9.0        0.0         0.0   
21855  Australia 2020-03-24       1081.0           9.0        0.0         0.0   
21856  Australia 2020-03-25       1081.0           9.0        0.0         0.0   
21857  Australia 2020-03-26       1081.0           9.0        0.0         0.0   

       population  total_vaccinations continent  death_rate  
21853    26177410                 0.0   Oceania        0.83  
21854    26177410                 0.0   Oceania        0.83  
21855    26177410                 0.0   Oceania        0.83  
21856    26177410                 0.0   Oceania        0.83  
21857    26177410                 0.0   Oceania        0.83  


/tmp/ipykernel_1023/637298485.py:22: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



## Distributed Processing (PySpark)

In [60]:
from pyspark.sql import SparkSession

def run_spark_analysis(df):
    spark = SparkSession.builder.appName("covid_pipeline").getOrCreate()
    sdf = spark.createDataFrame(df)

    sdf.createOrReplaceTempView("covid")

    result = spark.sql("""
        SELECT location,
               MAX(total_cases) AS max_cases,
               MAX(total_deaths) AS max_deaths,
               MAX(death_rate) AS death_rate
        FROM covid
        GROUP BY location
        ORDER BY max_cases DESC
        LIMIT 5
    """)

    result.show()
    return sdf

spark_df = run_spark_analysis(df)

+-------------+------------+----------+----------+
|     location|   max_cases|max_deaths|death_rate|
+-------------+------------+----------+----------+
|United States|1.03436829E8| 1193165.0|      6.12|
|        India| 4.5041748E7|  533623.0|      3.35|
|       France|  3.899749E7|  168091.0|     100.0|
|      Germany| 3.8437756E7|  174979.0|       5.2|
|       Brazil| 3.7511921E7|  702116.0|      6.93|
+-------------+------------+----------+----------+



## Data Warehouse (DuckDB)

In [61]:
import duckdb

def fix_death_rate(db_path="covid_warehouse.db"):
    conn = duckdb.connect(db_path)

    conn.execute("""
        UPDATE covid_stats
        SET death_rate = ROUND(
            LEAST(total_deaths * 100.0 / NULLIF(total_cases, 0), 100),
            2
        )
        WHERE total_cases > 0
    """)

    result = conn.execute("""
        SELECT location, MAX(death_rate) AS death_rate
        FROM covid_stats
        GROUP BY location
        ORDER BY death_rate DESC
    """).fetchdf()

    conn.close()

    print("Death rate updated")
    print(result)

fix_death_rate()

Death rate updated
         location  death_rate
0          France      100.00
1  United Kingdom      100.00
2         Germany      100.00
3           Italy       14.50
4          Canada        8.27
5          Brazil        6.93
6   United States        6.12
7       Australia        5.71
8           India        3.35


## SQL Analysis

In [62]:
def run_sql_queries(db_path="covid_warehouse.db"):
    conn = duckdb.connect(db_path)

    print("Top 5 countries:")
    print(conn.execute("""
        SELECT location, MAX(total_cases) AS total_cases
        FROM covid_stats
        GROUP BY location
        ORDER BY total_cases DESC
        LIMIT 5
    """).fetchdf())

    print("\nDeath rate:")
    print(conn.execute("""
        SELECT location, MAX(death_rate) AS death_rate
        FROM covid_stats
        GROUP BY location
        ORDER BY death_rate DESC
    """).fetchdf())

    conn.close()

run_sql_queries()

Top 5 countries:
        location  total_cases
0  United States  103436829.0
1          India   45041748.0
2         France   38997490.0
3        Germany   38437756.0
4         Brazil   37511921.0

Death rate:
         location  death_rate
0          France      100.00
1  United Kingdom      100.00
2         Germany      100.00
3           Italy       14.50
4          Canada        8.27
5          Brazil        6.93
6   United States        6.12
7       Australia        5.71
8           India        3.35


## Data Visualization

In [63]:
import plotly.express as px

def get_viz_data():
    conn = duckdb.connect('covid_warehouse.db')
    df_viz = conn.execute("""
        SELECT location,
               MAX(total_cases) AS total_cases,
               MAX(total_deaths) AS total_deaths,
               MAX(death_rate) AS death_rate
        FROM covid_stats
        GROUP BY location
        ORDER BY total_cases DESC
    """).fetchdf()
    conn.close()
    return df_viz

df_viz = get_viz_data()

fig1 = px.bar(
    df_viz,
    x='location',
    y='total_cases',
    title='Total Cases by Country',
    text='total_cases'
)

fig1.show()

In [64]:
fig2 = px.bar(
    df_viz,
    x='location',
    y='death_rate',
    title='Death Rate (%)',
    text='death_rate'
)

fig2.show()

In [65]:
def get_trend_data():
    conn = duckdb.connect('covid_warehouse.db')
    df_trend = conn.execute("""
        SELECT date, location, new_cases
        FROM covid_stats
        WHERE location IN ('United States', 'India', 'Brazil')
        AND new_cases > 0
        ORDER BY date
    """).fetchdf()
    conn.close()
    return df_trend

df_trend = get_trend_data()

fig3 = px.line(
    df_trend,
    x='date',
    y='new_cases',
    color='location',
    title='Daily New Cases'
)

fig3.show()

## Data Quality Checks

In [66]:
def run_quality_checks(df):
    checks = {
        "missing": df.isnull().sum().sum() == 0,
        "duplicates": df.duplicated().sum() == 0,
        "valid_dates": pd.to_datetime(df['date'], errors='coerce').notna().all(),
        "non_empty": len(df) > 0
    }

    for name, passed in checks.items():
        print(f"{name}: {'PASS' if passed else 'FAIL'}")

run_quality_checks(df)

missing: PASS
duplicates: PASS
valid_dates: PASS
non_empty: PASS


## Pipeline Orchestration (Airflow Design)

In [67]:
from datetime import datetime, timedelta

dag_config = {
    "dag_id": "covid_pipeline",
    "schedule": "@daily",
    "start_date": datetime(2026, 3, 17),
    "tasks": [
        "extract",
        "transform",
        "spark",
        "load_duckdb",
        "sql_analysis",
        "visualization",
        "quality_check"
    ]
}

print(dag_config)

{'dag_id': 'covid_pipeline', 'schedule': '@daily', 'start_date': datetime.datetime(2026, 3, 17, 0, 0), 'tasks': ['extract', 'transform', 'spark', 'load_duckdb', 'sql_analysis', 'visualization', 'quality_check']}
